# ДЗ 4. Операции, axis и broadcasting на данных «Дала Маркет»
Срок: **до понедельника 16.11, 19:00**.
Сдача: папка `hw04` репозитория `dsa-homework` (`hw04.ipynb`): `git add hw04` → `git commit` → `git push`, ссылку прикрепите в LMS.

Каждый ответ: **число + одна фраза для менеджера**. Циклы `for` по заказам не используем. Небольшой цикл по городам или категориям при сборке таблицы допустим.

- **[База]** обязательно для всех · **[Стандарт]** сколько получится · **[Челлендж]** для тех, кому легко.

## [Для всех] Чтение кода: без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
import numpy as np
# 1
a = np.array([2, 4, 6]); print(a / 2, a // 4)
1,0
2,1
3,1

# 2
m = np.ones((3, 4)); print(m.sum(axis=1))
4,4,4
# 3
print(np.arange(3) + np.arange(3).reshape(3, 1))
0 2 4
# 4
x = np.array([1, 4, 9]); print(np.sqrt(x).sum())
6
# 5
print((np.arange(1, 5) ** 2).mean())
6
```

Ваши ответы и объяснения:

1. 
2. 
3. 
4. 
5. 

In [15]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

orders.csv на месте: True


In [17]:
import csv
import numpy as np

CITY = {"алматы": "Алматы", "almaty": "Алматы", "астана": "Астана", "astana": "Астана",
        "шымкент": "Шымкент", "shymkent": "Шымкент", "караганда": "Караганда",
        "актобе": "Актобе", "атырау": "Атырау"}

with open("orders.csv", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

amount = np.array([float(r["amount"].replace(" ", "")) for r in rows])
paid   = np.array([r["status"] == "оплачен" for r in rows])
days   = np.array([float(r["delivery_days"]) if r["delivery_days"] else np.nan for r in rows])
city   = np.array([CITY.get(r["city"].strip().lower(), r["city"].strip()) for r in rows])
cust   = np.array([r["customer_id"] for r in rows])
month  = np.array([r["order_date"][:7] for r in rows])      # "2026-09" или "2026-10"
cat    = np.array([r["category"] for r in rows])
oid    = np.array([r["order_id"] for r in rows])

print(amount.shape, np.unique(month))

(5463,) ['2026-09' '2026-10']


In [19]:
a = np.array([2, 4, 6]); print(a / 2, a // 4)
m = np.ones((3, 4)); print(m.sum(axis=1))
print(np.arange(3) + np.arange(3).reshape(3, 1))


[1. 2. 3.] [0 1 1]
[4. 4. 4.]
[[0 1 2]
 [1 2 3]
 [2 3 4]]


In [21]:
# Чистые данные: без дублей (оставляем первое вхождение order_id), без TEST, только оплаченные
_, first = np.unique(oid, return_index=True)
keep = np.zeros(len(oid), dtype=bool)
keep[first] = True
clean = keep & (cust != "TEST") & paid
print("строк всего:", len(oid), "| в чистых данных:", clean.sum())

строк всего: 5463 | в чистых данных: 4739


## [База] 1. Выручка по чистым данным
Сколько строк осталось в `clean`? Выручка за сентябрь и октябрь и изменение в %. Совпадает ли с ДЗ 2?

In [26]:
# 1
print("Чистых строк:", clean.sum())

sep = amount[clean & (month == "2026-09")].sum()

octo = amount[clean & (month == "2026-10")].sum()

change = (octo - sep) / sep * 100

print("Сентябрь:", sep)
print("Октябрь:", octo)
print("Изменение:", round(change, 2), "%")

Чистых строк: 4739
Сентябрь: 35243410.0
Октябрь: 34419520.0
Изменение: -2.34 %


## [База] 2. Таблица «категория × месяц»
Соберите таблицу 5 × 2 по образцу таблицы городов с урока (категории: `np.unique(cat)`).
Какая категория приносит больше всего выручки за два месяца (`axis`)? Какая упала сильнее всех в %?

In [29]:
# 2
categories = np.unique(cat)
revenue = np.zeros((len(categories), 2))

for i, c in enumerate(categories):
    revenue[i, 0] = amount[clean & (cat == c) & (month == "2026-09")].sum()
    revenue[i, 1] = amount[clean & (cat == c) & (month == "2026-10")].sum()

total = revenue.sum(axis=1)
change = (revenue[:, 1] - revenue[:, 0]) / revenue[:, 0] * 100

print(categories)
print(revenue)
print("Больше всего выручки:", categories[np.argmax(total)])
print("Сильнее всего упала:", categories[np.argmin(change)])

for i, c in enumerate(categories):
    print(c, "Всего:", total[i], "Изменение:", round(change[i], 2), "%")

['Дом' 'Книги' 'Красота' 'Одежда' 'Электроника']
[[ 4997730.  5125790.]
 [ 1065300.  1140910.]
 [ 2610950.  2678990.]
 [ 9167100.  8505730.]
 [17402330. 16968100.]]
Больше всего выручки: Электроника
Сильнее всего упала: Одежда
Дом Всего: 10123520.0 Изменение: 2.56 %
Книги Всего: 2206210.0 Изменение: 7.1 %
Красота Всего: 5289940.0 Изменение: 2.61 %
Одежда Всего: 17672830.0 Изменение: -7.21 %
Электроника Всего: 34370430.0 Изменение: -2.5 %


## [База] 3. Средний чек по категориям в октябре
Соберите вторую таблицу: **число** заказов по категориям и месяцам (сумма маски, а не сумма amount).
Средний чек = выручка / число заказов, одной операцией над двумя таблицами. У какой категории он самый высокий?

In [32]:
# 3
orders_count = np.zeros((len(categories), 2))

for i, c in enumerate(categories):
    orders_count[i, 0] = (clean & (cat == c) & (month == "2026-09")).sum()
    orders_count[i, 1] = (clean & (cat == c) & (month == "2026-10")).sum()

avg_check = np.divide(
    revenue,
    orders_count,
    out=np.zeros_like(revenue),
    where=orders_count != 0
)

print("Количество заказов:")
print(orders_count)

print("Средний чек:")
print(avg_check)

avg_total = np.divide(
    revenue.sum(axis=1),
    orders_count.sum(axis=1),
    out=np.zeros(len(categories)),
    where=orders_count.sum(axis=1) != 0
)

print("Самый высокий средний чек:", categories[np.argmax(avg_total)])
print("Средний чек:", round(avg_total.max(), 2), "₸")

Количество заказов:
[[476. 476.]
 [265. 299.]
 [422. 435.]
 [678. 652.]
 [531. 505.]]
Средний чек:
[[10499.43277311 10768.46638655]
 [ 4020.          3815.75250836]
 [ 6187.08530806  6158.59770115]
 [13520.79646018 13045.59815951]
 [32772.74952919 33600.1980198 ]]
Самый высокий средний чек: Электроника
Средний чек: 33176.09 ₸


## [Стандарт] 4. Доля городов в выручке месяца
Используйте `table` городов с урока. Доля каждого города в выручке своего месяца: `table / table.sum(axis=0)`.
Проверьте, что каждый столбец в сумме даёт 1. Как изменилась доля Шымкента?

In [37]:
# 4
cities = np.unique(city)

table = np.zeros((len(cities), 2))

for i, c in enumerate(cities):
    table[i, 0] = amount[clean & (city == c) & (month == "2026-09")].sum()
    table[i, 1] = amount[clean & (city == c) & (month == "2026-10")].sum()

print(table)

[[ 2576960.  2540420.]
 [14501070. 14643020.]
 [ 9034220.  8850530.]
 [  923990.   969860.]
 [ 3648520.  3371680.]
 [ 4558650.  4044010.]]


In [39]:
share = table / table.sum(axis=0)

print(share)

[[0.07311892 0.07380754]
 [0.41145479 0.42542778]
 [0.25633785 0.25713694]
 [0.02621738 0.02817762]
 [0.10352347 0.09795837]
 [0.12934759 0.11749176]]


In [45]:
print(share.sum(axis=0))

shymkent = share[cities == "Шымкент"]

change = (shymkent[0, 1] - shymkent[0, 0]) * 100

print("Сентябрь:", round(shymkent[0, 0] * 100, 2), "%")
print("Октябрь:", round(shymkent[0, 1] * 100, 2), "%")
print("Изменение:", round(change, 2))

[1. 1.]
Сентябрь: 12.93 %
Октябрь: 11.75 %
Изменение: -1.19


## [Стандарт] 5. Кто дал падение
Вклад каждого города в общее падение выручки: изменение города / общее изменение. Сколько процентов падения даёт Шымкент?
Напишите менеджеру одну фразу: где искать причину.

In [47]:
# 5
change_city = table[:, 1] - table[:, 0]

print(cities)
print(change_city)

['Актобе' 'Алматы' 'Астана' 'Атырау' 'Караганда' 'Шымкент']
[ -36540.  141950. -183690.   45870. -276840. -514640.]


In [55]:
total_change = change_city.sum()
contribution = change_city / total_change * 100

for i, c in enumerate(cities):
    print(c, round(contribution[i], 2), "%")

shymkent = contribution[cities == "Шымкент"]

print("Вклад Шымкента:", round(shymkent[0], 2), "%")

Актобе 4.44 %
Алматы -17.23 %
Астана 22.3 %
Атырау -5.57 %
Караганда 33.6 %
Шымкент 62.46 %
Вклад Шымкента: 62.46 %


## [Стандарт] 6. Исправьте код от AI
AI написал код «доля сентября и октября внутри каждого города». Код падает. Объясните, почему, и исправьте.
```python
share = table / table.sum(axis=1)
```

In [ ]:
# 6

## [Челлендж] 7. Стандартизация суммы заказа
Для чистых оплаченных заказов посчитайте z-оценку: `z = (x - x.mean()) / x.std()`.
Сколько заказов с |z| > 3 и какая это доля? Проверьте, что у `z` среднее ≈ 0 и std = 1.

In [ ]:
# 7

## [Челлендж] 8. Масштабирование признаков (заготовка для ML)
Соберите матрицу `X` из двух столбцов: `amount` и `days` для чистых заказов без пропусков в `days` (`np.column_stack`).
Приведите каждый столбец к диапазону [0, 1] одной строкой: `(X - X.min(axis=0)) / (X.max(axis=0) - X.min(axis=0))`.
Проверьте min и max каждого столбца. Объясните своими словами, зачем модели нужны признаки в одном масштабе.

In [ ]:
# 8